# German Neural Reader – Google Colab

**Am Handy:** oben `Runtime` → **Run all / Alle ausführen**. Danach einmal die TXT-Datei auswählen. Alles Weitere läuft automatisch; am Ende wird die MP3 heruntergeladen.

Jeder neue Lauf löscht zuerst alte GermanReader-Ausgabedateien im aktuellen Colab-Runtime. Wenn ein Colab-Prozess noch aktiv läuft, musst du ihn in Colab selbst stoppen.

Verwendet die deutsche Stimme **Eva K**. Der Text wird in Google Colab verarbeitet, nicht lokal auf dem iPhone.

In [ ]:
#@title GermanReader starten – diese eine Zelle erledigt alles
SPEED = 0.94 #@param {type:"slider", min:0.78, max:1.18, step:0.01}
BITRATE = 40 #@param [32, 40, 64, 96] {type:"raw"}
SENTENCE_PAUSE = 0.24 #@param {type:"slider", min:0.10, max:0.80, step:0.02}

import os, pathlib, subprocess, urllib.request, sys, time
from google.colab import files

# Alte Ausgaben dieses Colab-Runtimes entfernen, damit jeder Lauf sauber startet.
for old in ['GermanReader.wav','GermanReader.mp3','GermanReader_input.txt']:
    try:
        pathlib.Path(old).unlink(missing_ok=True)
    except Exception:
        pass

print('1/5 · Umgebung wird vorbereitet …')
subprocess.run(['apt-get','-qq','update'], check=True)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg'], check=True)
subprocess.run([sys.executable,'-m','pip','-q','install','piper-tts==1.3.0'], check=True)

MODEL='de_DE-eva_k-x_low.onnx'
CONFIG='de_DE-eva_k-x_low.onnx.json'
BASE='https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/de/de_DE/eva_k/x_low/'

print('2/5 · Eva K wird geladen …')
if not pathlib.Path(MODEL).exists():
    urllib.request.urlretrieve(BASE+MODEL+'?download=true', MODEL)
if not pathlib.Path(CONFIG).exists():
    urllib.request.urlretrieve(BASE+CONFIG+'?download=true', CONFIG)

print('3/5 · Bitte jetzt GermanReader_input.txt auswählen …')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Keine Datei ausgewählt.')
name = next(iter(uploaded))
raw = uploaded[name]
try:
    text = raw.decode('utf-8')
except UnicodeDecodeError:
    text = raw.decode('utf-8-sig')
text = text.strip()
if not text:
    raise RuntimeError('Die TXT-Datei ist leer.')
print(f'{len(text):,} Zeichen geladen.')

print('4/5 · Sprache wird berechnet …')
t0=time.time()
cmd=[
    'piper','--model',MODEL,'--config',CONFIG,
    '--output_file','GermanReader.wav',
    '--length_scale',str(1.0/float(SPEED)),
    '--sentence_silence',str(float(SENTENCE_PAUSE))
]
p=subprocess.run(cmd,input=text,text=True,capture_output=True)
if p.returncode != 0:
    print(p.stderr[-3000:])
    raise RuntimeError('Piper konnte die Audiodatei nicht erzeugen.')
print(f'Sprachberechnung fertig nach {time.time()-t0:.1f} Sekunden.')

print('5/5 · MP3 wird erstellt …')
subprocess.run([
    'ffmpeg','-y','-hide_banner','-loglevel','error',
    '-i','GermanReader.wav','-codec:a','libmp3lame',
    '-b:a',f'{int(BITRATE)}k','GermanReader.mp3'
],check=True)
size=pathlib.Path('GermanReader.mp3').stat().st_size
print(f'Fertig · {size/1024/1024:.1f} MB · Download startet jetzt.')
files.download('GermanReader.mp3')
